# Regression Exercise
### Author: Xi Chen

We don't use the data from pollution_metadata.txt,because it is a reference for pollution_cleaneddata.csv.

In [2]:
import pandas as pd
import numpy as np
df=pd.read_csv('pollution_cleaneddata.csv')
x=df['POOR']
y=df['MORT']

In [3]:
def local_regression(x0, k):
    distance = abs(x - x0)
    nearest = distance.sort_values().head(k)

    x_local = x.loc[nearest.index].to_numpy(dtype=float)
    y_local = y.loc[nearest.index].to_numpy(dtype=float)
    distance_local = distance.loc[nearest.index].to_numpy(dtype=float)

    h = distance_local.max()

    if h == 0:
        weights = np.ones(k)
    else:
        weights = (
            1 - (distance_local / h) ** 3
        ) ** 3

    X_matrix = np.column_stack([
        np.ones(k),
        x_local
    ])

    W_matrix = np.diag(weights)

    beta = (
        np.linalg.pinv(
            X_matrix.T @ W_matrix @ X_matrix
        )
        @ X_matrix.T
        @ W_matrix
        @ y_local
    )

    beta0 = beta[0]
    beta1 = beta[1]

    x0_matrix = np.array([1, x0])

    y0 = x0_matrix @ beta

    y_local_predict = X_matrix @ beta
    residuals = y_local - y_local_predict

    sigma2 = (
        residuals @ residuals
    ) / (k - 2)

    covariance_beta = (
        sigma2
        * np.linalg.pinv(
            X_matrix.T @ X_matrix
        )
    )

    SE_y0 = np.sqrt(
        x0_matrix
        @ covariance_beta
        @ x0_matrix
    )

    print(
        f"When x0={x0} and k={k}, "
        f"the expected value is {y0:.3f} "
        f"and its standard error is {SE_y0:.3f}"
    )

    return (
        round(float(y0), 3),
        round(float(SE_y0), 3)
    )

In [4]:
local_regression(10, 15);
local_regression(18, 15);
local_regression(25, 15);

When x0=10 and k=15, the expected value is 899.794 and its standard error is 19.457
When x0=18 and k=15, the expected value is 956.156 and its standard error is 19.677
When x0=25 and k=15, the expected value is 1011.086 and its standard error is 26.056


The k nearest observations were weighted using a tricube kernel. Observations closer to the prediction point received larger weights.

The weighted local regression predicted mortality rates of 899.794, 956.156, and 1011.086 for areas where 10%, 18%, and 25% of families had an income below $3000, respectively. The predicted mortality rate increased as the percentage of low-income families increased, indicating a positive local association between POOR and MORT. The standard error was largest at POOR = 25%, suggesting greater uncertainty at this prediction point.